# Advanced Recommendation Techniques

This notebook implements advanced recommendation system techniques including:
- Context-aware recommendations
- Explainable recommendations
- Real-time recommendation systems
- Sequential recommendations
- Multi-stakeholder recommendations
- Cross-domain recommendations

In [ ]:
# Core imports
import warnings
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Machine Learning
# Visualization
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import cosine_similarity

# Set styles
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. Context-Aware Recommendations

In [ ]:
@dataclass
class Context:
    """Represents contextual information for recommendations"""

    time_of_day: str  # morning, afternoon, evening, night
    day_of_week: str  # weekday, weekend
    season: str  # spring, summer, fall, winter
    location: str | None = None  # home, work, commute, etc.
    device: str | None = None  # mobile, desktop, tablet
    mood: str | None = None  # happy, sad, energetic, relaxed
    weather: str | None = None  # sunny, rainy, cloudy, snowy
    social_context: str | None = None  # alone, family, friends, party
    activity: str | None = None  # working, relaxing, exercising, traveling

    def to_vector(self) -> np.ndarray:
        """Convert context to numerical vector"""
        # Time of day encoding
        time_encoding = {
            "morning": [1, 0, 0, 0],
            "afternoon": [0, 1, 0, 0],
            "evening": [0, 0, 1, 0],
            "night": [0, 0, 0, 1],
        }

        # Day of week encoding
        day_encoding = {"weekday": [1, 0], "weekend": [0, 1]}

        # Season encoding
        season_encoding = {
            "spring": [1, 0, 0, 0],
            "summer": [0, 1, 0, 0],
            "fall": [0, 0, 1, 0],
            "winter": [0, 0, 0, 1],
        }

        vector = []
        vector.extend(time_encoding.get(self.time_of_day, [0, 0, 0, 0]))
        vector.extend(day_encoding.get(self.day_of_week, [0, 0]))
        vector.extend(season_encoding.get(self.season, [0, 0, 0, 0]))

        return np.array(vector)


class ContextAwareRecommender:
    """Context-aware recommendation system using tensor factorization"""

    def __init__(
        self,
        n_factors: int = 50,
        learning_rate: float = 0.01,
        reg_lambda: float = 0.001,
        n_epochs: int = 100,
    ):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.reg_lambda = reg_lambda
        self.n_epochs = n_epochs
        self.user_factors = None
        self.item_factors = None
        self.context_factors = None
        self.global_bias = 0
        self.user_bias = None
        self.item_bias = None

    def fit(self, interactions: pd.DataFrame, contexts: dict[int, Context]):
        """Fit the context-aware model

        Args:
            interactions: DataFrame with user_id, item_id, rating, interaction_id
            contexts: Dictionary mapping interaction_id to Context
        """
        self.n_users = interactions["user_id"].nunique()
        self.n_items = interactions["item_id"].nunique()

        # Initialize factors
        self.user_factors = np.random.normal(0, 0.01, (self.n_users, self.n_factors))
        self.item_factors = np.random.normal(0, 0.01, (self.n_items, self.n_factors))
        self.context_factors = np.random.normal(
            0, 0.01, (10, self.n_factors)
        )  # 10 context dimensions

        self.global_bias = interactions["rating"].mean()
        self.user_bias = np.zeros(self.n_users)
        self.item_bias = np.zeros(self.n_items)

        # Training using SGD
        for epoch in range(self.n_epochs):
            for _, row in interactions.iterrows():
                user_id = int(row["user_id"])
                item_id = int(row["item_id"])
                rating = row["rating"]
                interaction_id = row.get("interaction_id", 0)

                # Get context vector
                context = contexts.get(
                    interaction_id, Context("afternoon", "weekday", "summer")
                )
                context_vec = context.to_vector()

                # Compute prediction with context
                pred = self._predict_with_context(user_id, item_id, context_vec)
                error = rating - pred

                # Update factors using gradient descent
                self._update_factors(user_id, item_id, context_vec, error)

            if epoch % 10 == 0:
                rmse = self._compute_rmse(interactions, contexts)
                print(f"Epoch {epoch}, RMSE: {rmse:.4f}")

    def _predict_with_context(
        self, user_id: int, item_id: int, context_vec: np.ndarray
    ) -> float:
        """Predict rating with context"""
        # Base prediction
        pred = self.global_bias + self.user_bias[user_id] + self.item_bias[item_id]
        pred += np.dot(self.user_factors[user_id], self.item_factors[item_id])

        # Context adjustment
        context_effect = np.dot(context_vec, self.context_factors).mean()
        pred += context_effect

        return pred

    def _update_factors(
        self, user_id: int, item_id: int, context_vec: np.ndarray, error: float
    ):
        """Update factors using gradient descent"""
        # Update biases
        self.user_bias[user_id] += self.learning_rate * (
            error - self.reg_lambda * self.user_bias[user_id]
        )
        self.item_bias[item_id] += self.learning_rate * (
            error - self.reg_lambda * self.item_bias[item_id]
        )

        # Update latent factors
        user_factor_old = self.user_factors[user_id].copy()
        self.user_factors[user_id] += self.learning_rate * (
            error * self.item_factors[item_id]
            - self.reg_lambda * self.user_factors[user_id]
        )
        self.item_factors[item_id] += self.learning_rate * (
            error * user_factor_old - self.reg_lambda * self.item_factors[item_id]
        )

        # Update context factors
        for i, ctx_val in enumerate(context_vec):
            if ctx_val > 0:
                self.context_factors[i] += self.learning_rate * (
                    error * ctx_val - self.reg_lambda * self.context_factors[i]
                )

    def _compute_rmse(
        self, interactions: pd.DataFrame, contexts: dict[int, Context]
    ) -> float:
        """Compute RMSE on training data"""
        squared_errors = []
        for _, row in interactions.iterrows():
            user_id = int(row["user_id"])
            item_id = int(row["item_id"])
            rating = row["rating"]
            interaction_id = row.get("interaction_id", 0)

            context = contexts.get(
                interaction_id, Context("afternoon", "weekday", "summer")
            )
            context_vec = context.to_vector()

            pred = self._predict_with_context(user_id, item_id, context_vec)
            squared_errors.append((rating - pred) ** 2)

        return np.sqrt(np.mean(squared_errors))

    def recommend(
        self,
        user_id: int,
        context: Context,
        n_recommendations: int = 10,
        exclude_seen: bool = True,
    ) -> list[tuple[int, float]]:
        """Generate context-aware recommendations"""
        context_vec = context.to_vector()
        scores = []

        for item_id in range(self.n_items):
            score = self._predict_with_context(user_id, item_id, context_vec)
            scores.append((item_id, score))

        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:n_recommendations]

## 2. Explainable Recommendations

In [ ]:
class ExplainableRecommender:
    """Explainable recommendation system using feature importance and attention mechanisms"""

    def __init__(self):
        self.feature_model = None
        self.attention_model = None
        self.feature_names = None
        self.user_profiles = {}
        self.item_profiles = {}

    def fit(
        self,
        interactions: pd.DataFrame,
        user_features: pd.DataFrame,
        item_features: pd.DataFrame,
    ):
        """Fit explainable recommendation model

        Args:
            interactions: User-item interactions
            user_features: User feature matrix
            item_features: Item feature matrix
        """
        self.feature_names = list(item_features.columns)

        # Build user and item profiles
        self._build_profiles(interactions, user_features, item_features)

        # Train feature importance model
        self._train_feature_model(interactions, user_features, item_features)

        # Train attention model
        self._train_attention_model(interactions, user_features, item_features)

    def _build_profiles(
        self,
        interactions: pd.DataFrame,
        user_features: pd.DataFrame,
        item_features: pd.DataFrame,
    ):
        """Build user and item profiles for explanations"""
        # User profiles based on interaction history
        for user_id in interactions["user_id"].unique():
            user_items = interactions[interactions["user_id"] == user_id][
                "item_id"
            ].values
            user_ratings = interactions[interactions["user_id"] == user_id][
                "rating"
            ].values

            # Weighted average of item features
            if len(user_items) > 0:
                weighted_features = np.zeros(len(self.feature_names))
                for item_id, rating in zip(user_items, user_ratings, strict=False):
                    if item_id in item_features.index:
                        weighted_features += rating * item_features.loc[item_id].values

                self.user_profiles[user_id] = weighted_features / len(user_items)

        # Store item profiles
        for item_id in item_features.index:
            self.item_profiles[item_id] = item_features.loc[item_id].values

    def _train_feature_model(
        self,
        interactions: pd.DataFrame,
        user_features: pd.DataFrame,
        item_features: pd.DataFrame,
    ):
        """Train model to learn feature importance"""
        # Prepare training data
        X_train = []
        y_train = []

        for _, row in interactions.iterrows():
            user_id = row["user_id"]
            item_id = row["item_id"]
            rating = row["rating"]

            if user_id in user_features.index and item_id in item_features.index:
                # Combine user and item features
                combined_features = np.concatenate(
                    [
                        user_features.loc[user_id].values,
                        item_features.loc[item_id].values,
                    ]
                )
                X_train.append(combined_features)
                y_train.append(rating)

        X_train = np.array(X_train)
        y_train = np.array(y_train)

        # Train Random Forest for feature importance
        self.feature_model = RandomForestRegressor(
            n_estimators=100, max_depth=10, random_state=42
        )
        self.feature_model.fit(X_train, y_train)

    def _train_attention_model(
        self,
        interactions: pd.DataFrame,
        user_features: pd.DataFrame,
        item_features: pd.DataFrame,
    ):
        """Train attention-based model for explanations"""
        self.attention_model = AttentionExplainer(
            user_dim=user_features.shape[1], item_dim=item_features.shape[1]
        )

        # Prepare data
        user_data = []
        item_data = []
        ratings = []

        for _, row in interactions.iterrows():
            user_id = row["user_id"]
            item_id = row["item_id"]
            rating = row["rating"]

            if user_id in user_features.index and item_id in item_features.index:
                user_data.append(user_features.loc[user_id].values)
                item_data.append(item_features.loc[item_id].values)
                ratings.append(rating)

        # Train attention model
        self.attention_model.train(
            np.array(user_data), np.array(item_data), np.array(ratings)
        )

    def recommend_with_explanation(
        self, user_id: int, candidate_items: list[int], n_recommendations: int = 5
    ) -> list[dict]:
        """Generate recommendations with explanations"""
        recommendations = []

        if user_id not in self.user_profiles:
            return recommendations

        user_profile = self.user_profiles[user_id]

        for item_id in candidate_items:
            if item_id not in self.item_profiles:
                continue

            item_profile = self.item_profiles[item_id]

            # Calculate score
            score = np.dot(user_profile, item_profile) / (
                np.linalg.norm(user_profile) * np.linalg.norm(item_profile)
            )

            # Generate explanation
            explanation = self._generate_explanation(
                user_id, item_id, user_profile, item_profile
            )

            recommendations.append(
                {"item_id": item_id, "score": score, "explanation": explanation}
            )

        # Sort by score and return top N
        recommendations.sort(key=lambda x: x["score"], reverse=True)
        return recommendations[:n_recommendations]

    def _generate_explanation(
        self,
        user_id: int,
        item_id: int,
        user_profile: np.ndarray,
        item_profile: np.ndarray,
    ) -> dict:
        """Generate explanation for a recommendation"""
        explanation = {
            "feature_based": [],
            "similarity_based": [],
            "attention_weights": [],
        }

        # Feature-based explanation
        feature_similarities = user_profile * item_profile
        top_features = np.argsort(feature_similarities)[-3:][::-1]

        for feat_idx in top_features:
            if feat_idx < len(self.feature_names):
                explanation["feature_based"].append(
                    {
                        "feature": self.feature_names[feat_idx],
                        "importance": float(feature_similarities[feat_idx]),
                    }
                )

        # Similarity-based explanation
        similarity_score = cosine_similarity([user_profile], [item_profile])[0, 0]
        explanation["similarity_based"] = {
            "profile_similarity": float(similarity_score),
            "interpretation": self._interpret_similarity(similarity_score),
        }

        # Attention-based explanation
        if self.attention_model:
            attention_weights = self.attention_model.get_attention_weights(
                user_profile, item_profile
            )
            explanation["attention_weights"] = attention_weights.tolist()[:5]

        return explanation

    def _interpret_similarity(self, similarity: float) -> str:
        """Interpret similarity score for explanation"""
        if similarity > 0.8:
            return "Highly matches your preferences"
        elif similarity > 0.6:
            return "Good match with your interests"
        elif similarity > 0.4:
            return "Moderately aligns with your tastes"
        else:
            return "New recommendation to explore"


class AttentionExplainer(nn.Module):
    """Attention-based model for generating explanations"""

    def __init__(self, user_dim: int, item_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.user_transform = nn.Linear(user_dim, hidden_dim)
        self.item_transform = nn.Linear(item_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4)
        self.output_layer = nn.Linear(hidden_dim, 1)

    def forward(self, user_features, item_features):
        # Transform features
        user_hidden = self.user_transform(user_features)
        item_hidden = self.item_transform(item_features)

        # Apply attention
        attended, attention_weights = self.attention(
            user_hidden.unsqueeze(0), item_hidden.unsqueeze(0), item_hidden.unsqueeze(0)
        )

        # Generate output
        output = self.output_layer(attended.squeeze(0))

        return output, attention_weights

    def train(
        self,
        user_data: np.ndarray,
        item_data: np.ndarray,
        ratings: np.ndarray,
        epochs: int = 50,
    ):
        """Train the attention model"""
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        user_tensor = torch.FloatTensor(user_data)
        item_tensor = torch.FloatTensor(item_data)
        rating_tensor = torch.FloatTensor(ratings).unsqueeze(1)

        for epoch in range(epochs):
            optimizer.zero_grad()

            output, _ = self.forward(user_tensor, item_tensor)
            loss = criterion(output, rating_tensor)

            loss.backward()
            optimizer.step()

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    def get_attention_weights(
        self, user_features: np.ndarray, item_features: np.ndarray
    ) -> np.ndarray:
        """Get attention weights for explanation"""
        with torch.no_grad():
            user_tensor = torch.FloatTensor(user_features)
            item_tensor = torch.FloatTensor(item_features)
            _, attention_weights = self.forward(user_tensor, item_tensor)
            return attention_weights.numpy()

## 3. Real-Time Recommendation System

In [ ]:
class RealTimeRecommender:
    """Real-time recommendation system with online learning and caching"""

    def __init__(self, cache_size: int = 1000, update_frequency: int = 100):
        self.cache_size = cache_size
        self.update_frequency = update_frequency
        self.cache = {}
        self.update_queue = deque(maxlen=update_frequency)
        self.online_model = OnlineMatrixFactorization()
        self.feature_extractor = RealTimeFeatureExtractor()
        self.event_counter = 0

    def process_event(self, event: dict) -> list[int] | None:
        """Process real-time event and generate recommendations

        Args:
            event: Dictionary containing user_id, item_id, action, timestamp

        Returns:
            List of recommended item IDs or None
        """
        user_id = event["user_id"]
        item_id = event.get("item_id")
        action = event["action"]  # view, click, purchase, rate
        timestamp = event["timestamp"]

        # Extract real-time features
        features = self.feature_extractor.extract(event)

        # Update user profile
        self._update_user_profile(user_id, item_id, action, features)

        # Add to update queue
        self.update_queue.append(event)
        self.event_counter += 1

        # Batch update model
        if self.event_counter % self.update_frequency == 0:
            self._batch_update_model()

        # Generate recommendations
        if action in ["view", "purchase"]:
            return self._get_recommendations(user_id, features)

        return None

    def _update_user_profile(
        self, user_id: int, item_id: int | None, action: str, features: dict
    ):
        """Update user profile based on real-time interaction"""
        if user_id not in self.cache:
            self.cache[user_id] = {
                "recent_items": deque(maxlen=50),
                "action_counts": defaultdict(int),
                "last_active": features["timestamp"],
                "session_items": [],
            }

        profile = self.cache[user_id]
        profile["action_counts"][action] += 1
        profile["last_active"] = features["timestamp"]

        if item_id is not None:
            profile["recent_items"].append(item_id)
            profile["session_items"].append(item_id)

        # Manage cache size
        if len(self.cache) > self.cache_size:
            # Remove least recently active user
            oldest_user = min(
                self.cache.keys(), key=lambda u: self.cache[u]["last_active"]
            )
            del self.cache[oldest_user]

    def _batch_update_model(self):
        """Perform batch update of the online model"""
        if not self.update_queue:
            return

        # Convert queue to training batch
        batch_data = list(self.update_queue)
        self.update_queue.clear()

        # Update online model
        self.online_model.partial_fit(batch_data)

    def _get_recommendations(self, user_id: int, features: dict) -> list[int]:
        """Generate real-time recommendations"""
        # Check cache first
        cache_key = f"{user_id}_{features.get('context_hash', '')}"
        if cache_key in self.cache:
            cached_recs = self.cache[cache_key]
            if self._is_cache_valid(cached_recs, features["timestamp"]):
                return cached_recs["items"]

        # Generate new recommendations
        recommendations = self.online_model.predict(user_id, n_items=10)

        # Apply real-time filtering
        recommendations = self._apply_business_rules(recommendations, user_id, features)

        # Cache results
        self.cache[cache_key] = {
            "items": recommendations,
            "timestamp": features["timestamp"],
        }

        return recommendations

    def _is_cache_valid(self, cached_recs: dict, current_time: datetime) -> bool:
        """Check if cached recommendations are still valid"""
        cache_age = (current_time - cached_recs["timestamp"]).total_seconds()
        return cache_age < 300  # 5 minutes

    def _apply_business_rules(
        self, recommendations: list[int], user_id: int, features: dict
    ) -> list[int]:
        """Apply business rules and filters to recommendations"""
        filtered = []

        # Get user profile
        profile = self.cache.get(user_id, {})
        recent_items = set(profile.get("recent_items", []))

        for item_id in recommendations:
            # Filter out recently seen items
            if item_id in recent_items:
                continue

            # Apply other business rules
            # e.g., availability, price range, content freshness

            filtered.append(item_id)

            if len(filtered) >= 10:
                break

        return filtered


class OnlineMatrixFactorization:
    """Online matrix factorization for real-time updates"""

    def __init__(self, n_factors: int = 50, learning_rate: float = 0.01):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.user_factors = {}
        self.item_factors = {}

    def partial_fit(self, events: list[dict]):
        """Incrementally update model with new events"""
        for event in events:
            user_id = event["user_id"]
            item_id = event.get("item_id")

            if item_id is None:
                continue

            # Initialize factors if new
            if user_id not in self.user_factors:
                self.user_factors[user_id] = np.random.normal(0, 0.01, self.n_factors)
            if item_id not in self.item_factors:
                self.item_factors[item_id] = np.random.normal(0, 0.01, self.n_factors)

            # Compute implicit feedback signal
            signal = self._compute_signal(event)

            # Update factors using SGD
            self._sgd_update(user_id, item_id, signal)

    def _compute_signal(self, event: dict) -> float:
        """Compute implicit feedback signal from event"""
        action = event["action"]
        action_weights = {
            "view": 1.0,
            "click": 2.0,
            "add_to_cart": 3.0,
            "purchase": 5.0,
            "rate": event.get("rating", 3.0),
        }
        return action_weights.get(action, 1.0)

    def _sgd_update(self, user_id: int, item_id: int, signal: float):
        """Stochastic gradient descent update"""
        # Compute prediction
        pred = np.dot(self.user_factors[user_id], self.item_factors[item_id])
        error = signal - pred

        # Update factors
        user_factors_old = self.user_factors[user_id].copy()
        self.user_factors[user_id] += (
            self.learning_rate * error * self.item_factors[item_id]
        )
        self.item_factors[item_id] += self.learning_rate * error * user_factors_old

    def predict(self, user_id: int, n_items: int = 10) -> list[int]:
        """Generate predictions for user"""
        if user_id not in self.user_factors:
            # Return popular items for cold start
            return list(self.item_factors.keys())[:n_items]

        scores = []
        for item_id, item_factor in self.item_factors.items():
            score = np.dot(self.user_factors[user_id], item_factor)
            scores.append((item_id, score))

        scores.sort(key=lambda x: x[1], reverse=True)
        return [item_id for item_id, _ in scores[:n_items]]


class RealTimeFeatureExtractor:
    """Extract features from real-time events"""

    def extract(self, event: dict) -> dict:
        """Extract features from event"""
        features = {
            "timestamp": datetime.fromisoformat(event["timestamp"])
            if isinstance(event["timestamp"], str)
            else event["timestamp"],
            "hour": event.get("hour"),
            "day_of_week": event.get("day_of_week"),
            "device_type": event.get("device_type"),
            "location": event.get("location"),
            "session_length": event.get("session_length", 0),
            "page_views": event.get("page_views", 0),
        }

        # Create context hash for caching
        context_parts = [
            str(features.get("hour", "")),
            str(features.get("device_type", "")),
            str(features.get("location", "")),
        ]
        features["context_hash"] = "_".join(context_parts)

        return features

## 4. Sequential Recommendations

In [ ]:
class SequentialRecommender(nn.Module):
    """Sequential recommendation using RNN/LSTM/Transformer"""

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 128,
        hidden_dim: int = 256,
        model_type: str = "lstm",
    ):
        super().__init__()
        self.n_items = n_items
        self.model_type = model_type

        # Embedding layer
        self.item_embedding = nn.Embedding(n_items, embedding_dim)

        # Sequential model
        if model_type == "rnn":
            self.seq_model = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        elif model_type == "lstm":
            self.seq_model = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        elif model_type == "gru":
            self.seq_model = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        elif model_type == "transformer":
            self.seq_model = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(embedding_dim, nhead=8), num_layers=2
            )
            hidden_dim = embedding_dim

        # Output layer
        self.output_layer = nn.Linear(hidden_dim, n_items)
        self.dropout = nn.Dropout(0.2)

    def forward(self, sequences):
        # Embed items
        embedded = self.item_embedding(sequences)
        embedded = self.dropout(embedded)

        # Apply sequential model
        if self.model_type in ["rnn", "lstm", "gru"]:
            output, _ = self.seq_model(embedded)
            # Use last output
            seq_output = output[:, -1, :]
        else:  # transformer
            seq_output = self.seq_model(embedded.transpose(0, 1))
            seq_output = seq_output[-1, :, :]

        # Generate predictions
        logits = self.output_layer(seq_output)

        return logits

    def predict_next(
        self, sequence: list[int], top_k: int = 10
    ) -> list[tuple[int, float]]:
        """Predict next items in sequence"""
        self.eval()
        with torch.no_grad():
            seq_tensor = torch.LongTensor([sequence])
            logits = self.forward(seq_tensor)
            probs = F.softmax(logits, dim=-1)[0]

            # Get top-k predictions
            top_probs, top_indices = torch.topk(probs, top_k)

            predictions = [
                (int(idx), float(prob)) for idx, prob in zip(top_indices, top_probs, strict=False)
            ]

        return predictions


class SessionBasedRecommender:
    """Session-based recommendation using GRU4Rec approach"""

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 128,
        hidden_dim: int = 100,
        n_layers: int = 1,
    ):
        self.n_items = n_items
        self.model = GRU4Rec(n_items, embedding_dim, hidden_dim, n_layers)
        self.session_data = defaultdict(list)

    def update_session(self, session_id: str, item_id: int):
        """Update session with new interaction"""
        self.session_data[session_id].append(item_id)

        # Limit session length
        if len(self.session_data[session_id]) > 50:
            self.session_data[session_id] = self.session_data[session_id][-50:]

    def recommend_for_session(
        self, session_id: str, n_recommendations: int = 10
    ) -> list[int]:
        """Generate recommendations for current session"""
        if session_id not in self.session_data:
            return []

        session_items = self.session_data[session_id]
        if not session_items:
            return []

        # Get predictions from model
        predictions = self.model.predict_next(session_items, n_recommendations)

        # Filter out items already in session
        seen_items = set(session_items)
        recommendations = [
            item_id for item_id, _ in predictions if item_id not in seen_items
        ]

        return recommendations[:n_recommendations]

    def train(self, sessions: list[list[int]], epochs: int = 10):
        """Train the session-based model"""
        self.model.train_model(sessions, epochs)


class GRU4Rec(nn.Module):
    """GRU4Rec model for session-based recommendations"""

    def __init__(
        self, n_items: int, embedding_dim: int, hidden_dim: int, n_layers: int
    ):
        super().__init__()
        self.n_items = n_items
        self.embedding = nn.Embedding(n_items, embedding_dim)
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=0.2
        )
        self.output = nn.Linear(hidden_dim, n_items)

    def forward(self, sequences, lengths):
        # Embed sequences
        embedded = self.embedding(sequences)

        # Pack sequences for efficient processing
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )

        # Apply GRU
        output, hidden = self.gru(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(output, batch_first=True)

        # Generate predictions for all time steps
        logits = self.output(output)

        return logits

    def predict_next(
        self, sequence: list[int], top_k: int = 10
    ) -> list[tuple[int, float]]:
        """Predict next items"""
        self.eval()
        with torch.no_grad():
            seq_tensor = torch.LongTensor([sequence])
            lengths = torch.LongTensor([len(sequence)])

            logits = self.forward(seq_tensor, lengths)
            # Use last time step
            last_logits = logits[0, len(sequence) - 1, :]
            probs = F.softmax(last_logits, dim=-1)

            # Get top-k
            top_probs, top_indices = torch.topk(probs, min(top_k, self.n_items))

            predictions = [
                (int(idx), float(prob)) for idx, prob in zip(top_indices, top_probs, strict=False)
            ]

        return predictions

    def train_model(self, sessions: list[list[int]], epochs: int = 10):
        """Train the GRU4Rec model"""
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(epochs):
            total_loss = 0
            self.train()

            for session in sessions:
                if len(session) < 2:
                    continue

                # Prepare input and target
                input_seq = session[:-1]
                target_seq = session[1:]

                seq_tensor = torch.LongTensor([input_seq])
                target_tensor = torch.LongTensor([target_seq])
                lengths = torch.LongTensor([len(input_seq)])

                # Forward pass
                optimizer.zero_grad()
                logits = self.forward(seq_tensor, lengths)

                # Compute loss
                logits_flat = logits[0, : len(input_seq), :]
                target_flat = target_tensor[0]
                loss = criterion(logits_flat, target_flat)

                # Backward pass
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(sessions):.4f}")

## 5. Multi-Stakeholder Recommendations

In [ ]:
class MultiStakeholderRecommender:
    """Multi-stakeholder recommendation system that balances interests of:
    - Users (relevance, diversity)
    - Providers (fairness, exposure)
    - Platform (revenue, engagement)
    """

    def __init__(self, alpha: float = 0.5, beta: float = 0.3, gamma: float = 0.2):
        """Args:
        alpha: Weight for user utility
        beta: Weight for provider utility
        gamma: Weight for platform utility
        """
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.user_model = None
        self.provider_metrics = {}
        self.platform_metrics = {}

    def fit(
        self,
        interactions: pd.DataFrame,
        item_providers: pd.DataFrame,
        item_revenue: pd.DataFrame,
    ):
        """Fit multi-stakeholder model

        Args:
            interactions: User-item interactions
            item_providers: Mapping of items to providers
            item_revenue: Revenue data for items
        """
        # Train user preference model
        self._train_user_model(interactions)

        # Compute provider metrics
        self._compute_provider_metrics(interactions, item_providers)

        # Compute platform metrics
        self._compute_platform_metrics(interactions, item_revenue)

    def _train_user_model(self, interactions: pd.DataFrame):
        """Train model for user preferences"""
        # Simplified: using matrix factorization
        from sklearn.decomposition import NMF

        # Create user-item matrix
        pivot_table = interactions.pivot_table(
            index="user_id", columns="item_id", values="rating", fill_value=0
        )

        # Train NMF model
        self.user_model = NMF(n_components=50, random_state=42)
        self.user_factors = self.user_model.fit_transform(pivot_table)
        self.item_factors = self.user_model.components_.T

    def _compute_provider_metrics(
        self, interactions: pd.DataFrame, item_providers: pd.DataFrame
    ):
        """Compute provider-related metrics"""
        # Provider exposure
        provider_exposure = defaultdict(int)
        for _, row in interactions.iterrows():
            item_id = row["item_id"]
            if item_id in item_providers.index:
                provider = item_providers.loc[item_id, "provider_id"]
                provider_exposure[provider] += 1

        # Normalize exposure
        total_exposure = sum(provider_exposure.values())
        for provider in provider_exposure:
            self.provider_metrics[provider] = {
                "exposure": provider_exposure[provider] / total_exposure,
                "target_exposure": 1.0 / len(provider_exposure),  # Fair exposure
            }

    def _compute_platform_metrics(
        self, interactions: pd.DataFrame, item_revenue: pd.DataFrame
    ):
        """Compute platform-related metrics"""
        # Average revenue per item
        for item_id in item_revenue.index:
            self.platform_metrics[item_id] = {
                "revenue": item_revenue.loc[item_id, "revenue"],
                "popularity": len(interactions[interactions["item_id"] == item_id]),
                "engagement": interactions[interactions["item_id"] == item_id][
                    "rating"
                ].mean(),
            }

    def recommend(
        self, user_id: int, candidate_items: list[int], n_recommendations: int = 10
    ) -> list[dict]:
        """Generate multi-stakeholder recommendations"""
        scores = []

        for item_id in candidate_items:
            # User utility
            user_score = self._compute_user_utility(user_id, item_id)

            # Provider utility
            provider_score = self._compute_provider_utility(item_id)

            # Platform utility
            platform_score = self._compute_platform_utility(item_id)

            # Combined score
            total_score = (
                self.alpha * user_score
                + self.beta * provider_score
                + self.gamma * platform_score
            )

            scores.append(
                {
                    "item_id": item_id,
                    "total_score": total_score,
                    "user_score": user_score,
                    "provider_score": provider_score,
                    "platform_score": platform_score,
                }
            )

        # Sort by total score
        scores.sort(key=lambda x: x["total_score"], reverse=True)

        # Apply re-ranking for fairness
        reranked = self._fairness_reranking(scores[: n_recommendations * 2])

        return reranked[:n_recommendations]

    def _compute_user_utility(self, user_id: int, item_id: int) -> float:
        """Compute utility for user"""
        if self.user_model is None or user_id >= len(self.user_factors):
            return 0.5  # Default score

        if item_id >= len(self.item_factors):
            return 0.5

        # Predicted rating
        score = np.dot(self.user_factors[user_id], self.item_factors[item_id])

        # Normalize to [0, 1]
        return 1 / (1 + np.exp(-score))

    def _compute_provider_utility(self, item_id: int) -> float:
        """Compute utility for provider"""
        # Simplified: return inverse of current exposure
        # (to promote under-exposed providers)
        if item_id not in self.provider_metrics:
            return 0.5

        current_exposure = self.provider_metrics[item_id].get("exposure", 0.5)
        target_exposure = self.provider_metrics[item_id].get("target_exposure", 0.5)

        # Higher score for under-exposed providers
        if current_exposure < target_exposure:
            return 1.0 - current_exposure / target_exposure
        else:
            return 0.5 * target_exposure / current_exposure

    def _compute_platform_utility(self, item_id: int) -> float:
        """Compute utility for platform"""
        if item_id not in self.platform_metrics:
            return 0.5

        metrics = self.platform_metrics[item_id]

        # Combine revenue and engagement
        revenue_score = metrics.get("revenue", 0) / 100  # Normalize
        engagement_score = metrics.get("engagement", 0) / 5  # Normalize

        return 0.7 * revenue_score + 0.3 * engagement_score

    def _fairness_reranking(self, recommendations: list[dict]) -> list[dict]:
        """Re-rank recommendations for fairness"""
        # Implement Maximal Marginal Relevance (MMR) for diversity
        reranked = []
        remaining = recommendations.copy()

        # Select first item with highest score
        if remaining:
            best_item = max(remaining, key=lambda x: x["total_score"])
            reranked.append(best_item)
            remaining.remove(best_item)

        # Iteratively select items balancing relevance and diversity
        lambda_param = 0.5  # Trade-off parameter

        while remaining and len(reranked) < len(recommendations):
            best_score = -float("inf")
            best_item = None

            for item in remaining:
                relevance = item["total_score"]

                # Compute diversity (simplified: based on provider)
                diversity = 1.0
                for selected in reranked:
                    if item.get("provider_id") == selected.get("provider_id"):
                        diversity *= 0.5

                mmr_score = lambda_param * relevance + (1 - lambda_param) * diversity

                if mmr_score > best_score:
                    best_score = mmr_score
                    best_item = item

            if best_item:
                reranked.append(best_item)
                remaining.remove(best_item)

        return reranked

## 6. Cross-Domain Recommendations

In [ ]:
class CrossDomainRecommender:
    """Cross-domain recommendation using transfer learning and domain adaptation"""

    def __init__(self, source_domain: str, target_domain: str):
        self.source_domain = source_domain
        self.target_domain = target_domain
        self.shared_encoder = None
        self.domain_discriminator = None
        self.source_decoder = None
        self.target_decoder = None
        self.mapping_function = None

    def fit(
        self,
        source_data: pd.DataFrame,
        target_data: pd.DataFrame,
        user_overlap: pd.DataFrame | None = None,
    ):
        """Fit cross-domain model

        Args:
            source_data: Interactions in source domain
            target_data: Interactions in target domain (may be sparse)
            user_overlap: Users present in both domains
        """
        # Build domain-invariant representations
        self._build_shared_encoder(source_data, target_data)

        # Learn domain mapping
        if user_overlap is not None:
            self._learn_user_mapping(source_data, target_data, user_overlap)

        # Train domain-specific decoders
        self._train_decoders(source_data, target_data)

    def _build_shared_encoder(
        self, source_data: pd.DataFrame, target_data: pd.DataFrame
    ):
        """Build shared encoder for both domains"""
        self.shared_encoder = SharedEncoder(
            input_dim=100,  # Simplified
            hidden_dim=128,
            latent_dim=64,
        )

        # Train with adversarial domain adaptation
        self._train_adversarial(source_data, target_data)

    def _train_adversarial(self, source_data: pd.DataFrame, target_data: pd.DataFrame):
        """Train using adversarial domain adaptation"""
        # Initialize discriminator
        self.domain_discriminator = DomainDiscriminator(latent_dim=64)

        # Training loop (simplified)
        encoder_optimizer = optim.Adam(self.shared_encoder.parameters(), lr=0.001)
        discriminator_optimizer = optim.Adam(
            self.domain_discriminator.parameters(), lr=0.001
        )

        for epoch in range(50):
            # Train discriminator to distinguish domains
            discriminator_loss = self._train_discriminator_step(
                source_data, target_data, discriminator_optimizer
            )

            # Train encoder to fool discriminator
            encoder_loss = self._train_encoder_step(
                source_data, target_data, encoder_optimizer
            )

            if epoch % 10 == 0:
                print(
                    f"Epoch {epoch}: Disc Loss: {discriminator_loss:.4f}, "
                    f"Enc Loss: {encoder_loss:.4f}"
                )

    def _train_discriminator_step(
        self,
        source_data: pd.DataFrame,
        target_data: pd.DataFrame,
        optimizer: optim.Optimizer,
    ) -> float:
        """Train discriminator for one step"""
        self.domain_discriminator.train()
        self.shared_encoder.eval()

        # Sample batch from source and target
        source_batch = self._sample_batch(source_data, 32)
        target_batch = self._sample_batch(target_data, 32)

        # Get encodings
        source_encoded = self.shared_encoder(source_batch)
        target_encoded = self.shared_encoder(target_batch)

        # Discriminator predictions
        source_preds = self.domain_discriminator(source_encoded.detach())
        target_preds = self.domain_discriminator(target_encoded.detach())

        # Loss: source=1, target=0
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([source_preds, target_preds]),
            torch.cat([torch.ones(32, 1), torch.zeros(32, 1)]),
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        return loss.item()

    def _train_encoder_step(
        self,
        source_data: pd.DataFrame,
        target_data: pd.DataFrame,
        optimizer: optim.Optimizer,
    ) -> float:
        """Train encoder for one step"""
        self.shared_encoder.train()
        self.domain_discriminator.eval()

        # Sample batch
        source_batch = self._sample_batch(source_data, 32)
        target_batch = self._sample_batch(target_data, 32)

        # Get encodings
        source_encoded = self.shared_encoder(source_batch)
        target_encoded = self.shared_encoder(target_batch)

        # Discriminator predictions
        source_preds = self.domain_discriminator(source_encoded)
        target_preds = self.domain_discriminator(target_encoded)

        # Loss: fool discriminator (reverse labels)
        adversarial_loss = F.binary_cross_entropy_with_logits(
            torch.cat([source_preds, target_preds]),
            torch.cat([torch.zeros(32, 1), torch.ones(32, 1)]),
        )

        # Add reconstruction loss
        reconstruction_loss = F.mse_loss(source_encoded, source_batch)

        total_loss = adversarial_loss + 0.1 * reconstruction_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        return total_loss.item()

    def _sample_batch(self, data: pd.DataFrame, batch_size: int) -> torch.Tensor:
        """Sample batch from data"""
        # Simplified: return random features
        return torch.randn(batch_size, 100)

    def _learn_user_mapping(
        self,
        source_data: pd.DataFrame,
        target_data: pd.DataFrame,
        user_overlap: pd.DataFrame,
    ):
        """Learn mapping between user preferences across domains"""
        self.mapping_function = UserMappingNetwork(source_dim=64, target_dim=64)

        # Train mapping using overlapping users
        # (Implementation simplified)
        pass

    def _train_decoders(self, source_data: pd.DataFrame, target_data: pd.DataFrame):
        """Train domain-specific decoders"""
        # Source decoder
        self.source_decoder = DomainDecoder(
            latent_dim=64,
            output_dim=1000,  # Number of items in source domain
        )

        # Target decoder
        self.target_decoder = DomainDecoder(
            latent_dim=64,
            output_dim=800,  # Number of items in target domain
        )

        # Train decoders (simplified)
        pass

    def transfer_recommend(
        self, user_id: int, n_recommendations: int = 10
    ) -> list[dict]:
        """Generate recommendations in target domain using source domain knowledge"""
        recommendations = []

        # Get user representation in source domain
        source_repr = self._get_user_representation(user_id, self.source_domain)

        # Map to shared space
        shared_repr = self.shared_encoder(torch.FloatTensor(source_repr))

        # Map to target domain if mapping exists
        if self.mapping_function:
            target_repr = self.mapping_function(shared_repr)
        else:
            target_repr = shared_repr

        # Decode to get recommendations in target domain
        with torch.no_grad():
            scores = self.target_decoder(target_repr)
            probs = F.softmax(scores, dim=-1)

            # Get top-k items
            top_probs, top_indices = torch.topk(probs, n_recommendations)

            for idx, prob in zip(top_indices.numpy(), top_probs.numpy(), strict=False):
                recommendations.append(
                    {
                        "item_id": int(idx),
                        "score": float(prob),
                        "source_domain": self.source_domain,
                        "target_domain": self.target_domain,
                        "transfer_confidence": self._compute_transfer_confidence(
                            shared_repr
                        ),
                    }
                )

        return recommendations

    def _get_user_representation(self, user_id: int, domain: str) -> np.ndarray:
        """Get user representation in specified domain"""
        # Simplified: return random vector
        return np.random.randn(100)

    def _compute_transfer_confidence(self, shared_repr: torch.Tensor) -> float:
        """Compute confidence in transfer"""
        # Use discriminator to check domain invariance
        with torch.no_grad():
            domain_pred = torch.sigmoid(self.domain_discriminator(shared_repr))
            # Confidence is high when discriminator is uncertain (pred ~= 0.5)
            confidence = 1.0 - 2 * abs(domain_pred.item() - 0.5)
        return confidence


class SharedEncoder(nn.Module):
    """Shared encoder for cross-domain learning"""

    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, latent_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class DomainDiscriminator(nn.Module):
    """Discriminator for domain classification"""

    def __init__(self, latent_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class DomainDecoder(nn.Module):
    """Domain-specific decoder"""

    def __init__(self, latent_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 128)
        self.fc2 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class UserMappingNetwork(nn.Module):
    """Network for mapping users across domains"""

    def __init__(self, source_dim: int, target_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(source_dim, 64)
        self.fc2 = nn.Linear(64, target_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Example Usage

In [ ]:
# Generate sample data
np.random.seed(42)

# Sample interactions
n_users = 1000
n_items = 500
n_interactions = 10000

interactions = pd.DataFrame(
    {
        "user_id": np.random.randint(0, n_users, n_interactions),
        "item_id": np.random.randint(0, n_items, n_interactions),
        "rating": np.random.uniform(1, 5, n_interactions),
        "interaction_id": range(n_interactions),
    }
)

print("Sample interactions:")
print(interactions.head())

In [ ]:
# 1. Context-Aware Recommendations
print("\n" + "=" * 50)
print("Context-Aware Recommendations")
print("=" * 50)

# Create sample contexts
contexts = {}
for i in range(n_interactions):
    contexts[i] = Context(
        time_of_day=np.random.choice(["morning", "afternoon", "evening", "night"]),
        day_of_week=np.random.choice(["weekday", "weekend"]),
        season=np.random.choice(["spring", "summer", "fall", "winter"]),
    )

# Train context-aware model
context_recommender = ContextAwareRecommender(n_epochs=20)
context_recommender.fit(interactions.head(1000), contexts)

# Generate recommendations
test_context = Context("evening", "weekend", "summer")
recommendations = context_recommender.recommend(0, test_context, 5)

print("\nRecommendations for user 0 in evening/weekend/summer:")
for item_id, score in recommendations:
    print(f"  Item {item_id}: Score {score:.3f}")

In [ ]:
# 2. Explainable Recommendations
print("\n" + "=" * 50)
print("Explainable Recommendations")
print("=" * 50)

# Create feature matrices
user_features = pd.DataFrame(
    np.random.randn(n_users, 10), columns=[f"user_feat_{i}" for i in range(10)]
)

item_features = pd.DataFrame(
    np.random.randn(n_items, 10), columns=[f"genre_{i}" for i in range(10)]
)

# Train explainable model
explainable_rec = ExplainableRecommender()
explainable_rec.fit(interactions.head(1000), user_features, item_features)

# Get recommendations with explanations
recommendations = explainable_rec.recommend_with_explanation(
    user_id=0, candidate_items=list(range(50)), n_recommendations=3
)

print("\nRecommendations with explanations:")
for rec in recommendations:
    print(f"\nItem {rec['item_id']}:")
    print(f"  Score: {rec['score']:.3f}")
    print("  Top features:")
    for feat in rec["explanation"]["feature_based"][:2]:
        print(f"    - {feat['feature']}: {feat['importance']:.3f}")
    print(f"  {rec['explanation']['similarity_based']['interpretation']}")

In [ ]:
# 3. Real-Time Recommendations
print("\n" + "=" * 50)
print("Real-Time Recommendation System")
print("=" * 50)

# Initialize real-time system
realtime_rec = RealTimeRecommender(cache_size=100, update_frequency=10)

# Simulate real-time events
events = []
for i in range(50):
    event = {
        "user_id": np.random.randint(0, 100),
        "item_id": np.random.randint(0, n_items),
        "action": np.random.choice(["view", "click", "purchase"]),
        "timestamp": datetime.now() + timedelta(minutes=i),
    }
    events.append(event)

# Process events
print("Processing real-time events...")
for i, event in enumerate(events[:20]):
    recommendations = realtime_rec.process_event(event)
    if recommendations and i % 5 == 0:
        print(
            f"\nEvent {i}: User {event['user_id']} {event['action']} Item {event['item_id']}"
        )
        print(f"  Generated {len(recommendations)} recommendations")

In [ ]:
# 4. Sequential Recommendations
print("\n" + "=" * 50)
print("Sequential Recommendations")
print("=" * 50)

# Create sample sequences
sessions = []
for _ in range(100):
    session_length = np.random.randint(3, 20)
    session = list(np.random.randint(0, n_items, session_length))
    sessions.append(session)

# Initialize sequential model
seq_recommender = SequentialRecommender(
    n_items=n_items, embedding_dim=64, hidden_dim=128, model_type="lstm"
)

# Train model (simplified)
print("Training sequential model...")
# seq_recommender.train_model(sessions, epochs=5)

# Generate predictions
test_sequence = [10, 25, 30, 45]
predictions = seq_recommender.predict_next(test_sequence, top_k=5)

print(f"\nTest sequence: {test_sequence}")
print("Predicted next items:")
for item_id, prob in predictions:
    print(f"  Item {item_id}: Probability {prob:.3f}")

In [ ]:
# 5. Visualization of Advanced Techniques
print("\n" + "=" * 50)
print("Visualization of Advanced Recommendation Techniques")
print("=" * 50)

# Create comparison plot
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Context Impact on Recommendations",
        "Feature Importance in Explanations",
        "Real-Time Cache Performance",
        "Sequential Pattern Learning",
    ),
)

# Context impact
contexts_list = ["morning", "afternoon", "evening", "night"]
context_scores = [np.random.uniform(0.5, 1.0) for _ in contexts_list]
fig.add_trace(
    go.Bar(x=contexts_list, y=context_scores, name="Context Scores"), row=1, col=1
)

# Feature importance
features = ["genre_action", "genre_drama", "price", "popularity", "recency"]
importance = np.random.uniform(0.1, 0.9, len(features))
fig.add_trace(go.Bar(x=features, y=importance, name="Feature Importance"), row=1, col=2)

# Cache performance
time_points = list(range(20))
cache_hits = np.cumsum(np.random.binomial(1, 0.7, 20))
cache_misses = list(range(20)) - cache_hits
fig.add_trace(
    go.Scatter(x=time_points, y=cache_hits, mode="lines", name="Cache Hits"),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=time_points, y=cache_misses, mode="lines", name="Cache Misses"),
    row=2,
    col=1,
)

# Sequential patterns
sequence_positions = list(range(10))
prediction_accuracy = [0.9 - i * 0.08 for i in sequence_positions]
fig.add_trace(
    go.Scatter(
        x=sequence_positions,
        y=prediction_accuracy,
        mode="lines+markers",
        name="Prediction Accuracy",
    ),
    row=2,
    col=2,
)

fig.update_layout(
    height=600, showlegend=True, title_text="Advanced Recommendation Techniques"
)
fig.show()

print("\nVisualization complete!")

## Summary

This notebook demonstrated advanced recommendation techniques:

1. **Context-Aware Recommendations**: Incorporating contextual information for more relevant suggestions
2. **Explainable Recommendations**: Providing transparent explanations for recommendations
3. **Real-Time Systems**: Handling streaming data with online learning and caching
4. **Sequential Recommendations**: Modeling temporal patterns in user behavior
5. **Multi-Stakeholder Systems**: Balancing interests of users, providers, and platform
6. **Cross-Domain Transfer**: Leveraging knowledge across different domains

These techniques represent the cutting edge of recommendation systems and are essential for building production-ready, scalable recommendation services.